In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

In [ ]:
tickers = {
    "SPX": "^GSPC",
    "TNX": "^TNX",
    "GOLD": "GC=F",
    "VIX": "^VIX",
    "DXY": "DX-Y.NYB"
}

start = "1995-12-01"
end   = "2026-12-31"

data = {}

for name, ticker in tickers.items():
    df = yf.download(
        ticker,
        start=start,
        end=end,
        progress=False
    )
    
    data[name] = df["Close"]

prices = pd.concat(data, axis=1)
prices.columns = tickers.keys()

prices["TNX"] = prices["TNX"] / 10
prices = prices.sort_index()

# Make calendar daily
full_index = pd.date_range(
    start=prices.index.min(),
    end=prices.index.max(),
    freq="D"
)

prices = prices.reindex(full_index)

# Information set carry-forward
prices = prices.ffill()

prices = prices.loc["1996-01-01":"2026-12-31"]

prices = prices.reset_index()
prices = prices.rename(columns={"index":"date"})

In [ ]:
xauusd = pd.read_csv("dataset/XAUUSD.csv")
xauusd = xauusd[['Tanggal', 'Terakhir']]
xauusd = xauusd.rename(columns={
    'Tanggal': 'date',
    'Terakhir': 'GOLD'
})
xauusd['GOLD'] = (xauusd['GOLD']
                   .str.replace('"', '')
                   .str.replace(',', '.')
                   .astype(float))

xauusd['date'] = (xauusd['date']
                  .str.replace('"', '')
                  .pipe(pd.to_datetime, format='%d/%m/%Y'))

xauusd = xauusd.set_index('date').sort_index()
full_range = pd.date_range(start=xauusd.index.min(), end=xauusd.index.max(), freq='D')
xauusd = xauusd.reindex(full_range).ffill().reset_index()
xauusd = xauusd.rename(columns={'index': 'date'})

xauusd = xauusd[(xauusd['date'] >= "1996-01-01") & (xauusd['date'] <= '2000-08-29')]
xauusd = xauusd.sort_values(by='date', ascending=True)
xauusd

In [ ]:
import pandas as pd

xauusd['date'] = pd.to_datetime(xauusd['date'])
xauusd = xauusd.set_index('date')

prices = prices.set_index('date')

prices['GOLD'] = prices['GOLD'].combine_first(xauusd['GOLD'])

prices = prices.reset_index().rename(columns={'index': 'date'})

In [ ]:
prices

In [ ]:
prices.to_csv("dataset/price_action.csv", index=False)